# Bronze inventory
Connect to the platform MinIO S3 API as described in README.md. Use the inventory table to explain what landed, then open a collection to show the actual records.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from bronze_review import decode_file, inventory, open_bronze

bronze = open_bronze()
catalog = inventory(bronze)
print(f"Objects listed: {len(bronze.objects)}")

In [ ]:
columns = [
    "kind", "source", "collection", "key", "bytes", "rows", "status", "fields",
    "run_id", "published_at", "generation", "partition", "offset", "description",
]
inventory_df = pd.DataFrame.from_records(catalog, columns=columns)
inventory_df.to_csv(Path("bronze_inventory.csv"), index=False, encoding="utf-8-sig")

print("Dataset inventory")
display(inventory_df[["kind", "source", "collection", "rows", "status", "description"]])
exceptions_df = inventory_df[inventory_df["status"].fillna("ok").ne("ok")]
if not exceptions_df.empty:
    print("Inventory exceptions")
    display(exceptions_df[["key", "status"]])

Batch rows are counted per snapshot and should not be summed to derive unique business records. Kafka rows represent deliveries, each identified by generation, partition, and offset. Unclassified objects remain in MinIO and the inventory for manual inspection.

## Open the actual records
The default example shows the latest dealer customer snapshot. Change the source and collection to show another snapshot, for example `erpnext` and `Sales Invoice`. For an inbound CSV, JSON file, or Kafka envelope, copy its full key from `inventory_df` or `bronze_inventory.csv` into `object_key`. One row represents one source record. Nested ERP line items stay with their parent record.

In [ ]:
source = "dealer-sql"
collection = "customers"
object_key = ""

if object_key:
    selected = inventory_df.loc[inventory_df["key"].eq(object_key)]
else:
    snapshots = inventory_df.loc[
        inventory_df["kind"].eq("snapshot") & inventory_df["source"].eq(source)
    ]
    latest_run = snapshots.sort_values(["published_at", "run_id"]).tail(1)["run_id"]
    selected = snapshots.loc[
        snapshots["run_id"].isin(latest_run) & snapshots["collection"].eq(collection)
    ]

if len(selected) != 1:
    raise ValueError("Select a source and collection from inventory_df, or enter one exact object key.")
entry = selected.iloc[0]
if entry["status"] != "ok":
    raise ValueError(f"Choose a readable data object. Selected status: {entry['status']}")
records, fields = decode_file(entry["key"], bronze.read(entry["key"]))
if records is None:
    raise ValueError("Choose a JSON, JSONL, or CSV object to display as a DataFrame.")

records_df = pd.json_normalize(records, max_level=1) if records else pd.DataFrame(columns=fields)
print(f"{entry['source']} | {entry['collection']} | {len(records_df)} records")
print(entry["description"])
display(records_df.head(20))